In [28]:
import pickle
import pandas as pd
from pathlib import Path


with open("processed_battery_features.pkl", "rb") as f:
    feature_df = pickle.load(f)

cell_ids = feature_df["cell_id"].unique().tolist()

print(f"{len(cell_ids)} cells found")
print(cell_ids[:10])

130 cells found
['Tongji2_CY25-05_1--10', 'Tongji1_CY45-05_1--26', 'Tongji3_CY25-05_4--3', 'Tongji1_CY25-025_1--7', 'Tongji2_CY45-05_1--16', 'Tongji2_CY45-05_1--23', 'Tongji1_CY45-05_1--20', 'Tongji2_CY45-05_1--24', 'Tongji2_CY25-05_1--3', 'Tongji1_CY45-05_1--1']


In [29]:
print(feature_df.shape)
feature_df.head()

(3806, 11)


,cell_id,cycle_number,SOH,I_mean,I_std,charge_duration,discharge_duration,V_mean,V_std,DTW_I,DTW_V
0,Tongji2_CY25-05_1--10,2,1.000261,-0.476854,2.234250,8525.136719,3396.996094,3.720052,0.504689,0.032572,0.054916
1,Tongji2_CY25-05_1--10,3,1.000000,-0.473836,2.234559,8519.181641,3396.160156,3.719458,0.505072,0.000000,0.000000
2,Tongji2_CY25-05_1--10,4,0.999358,-0.474358,2.235798,8516.205078,3394.027344,3.718133,0.505344,0.018442,0.083316
3,Tongji2_CY25-05_1--10,5,0.998792,-0.470999,2.234788,8514.187500,3392.152344,3.718795,0.505500,0.031987,0.057441
4,Tongji2_CY25-05_1--10,10,0.994373,-0.470528,2.235101,8478.203125,3377.656250,3.718256,0.504946,0.036931,0.067132


In [30]:
data_folder = Path("Battery_life_Dataset/Tongji")

datasets = []

for pkl_file in data_folder.glob("*.pkl"):

    with open(pkl_file, "rb") as f:
        data = pickle.load(f)

    if data["cell_id"] in cell_ids:
        datasets.append(data)

print(f"Loaded {len(datasets)} datasets")

Loaded 130 datasets


In [31]:
loaded_cell_ids = [d["cell_id"] for d in datasets]

print(sorted(loaded_cell_ids))

['Tongji1_CY25-025_1--1', 'Tongji1_CY25-025_1--2', 'Tongji1_CY25-025_1--3', 'Tongji1_CY25-025_1--4', 'Tongji1_CY25-025_1--5', 'Tongji1_CY25-025_1--6', 'Tongji1_CY25-025_1--7', 'Tongji1_CY25-05_1--1', 'Tongji1_CY25-05_1--10', 'Tongji1_CY25-05_1--11', 'Tongji1_CY25-05_1--12', 'Tongji1_CY25-05_1--13', 'Tongji1_CY25-05_1--14', 'Tongji1_CY25-05_1--15', 'Tongji1_CY25-05_1--16', 'Tongji1_CY25-05_1--17', 'Tongji1_CY25-05_1--18', 'Tongji1_CY25-05_1--19', 'Tongji1_CY25-05_1--2', 'Tongji1_CY25-05_1--3', 'Tongji1_CY25-05_1--4', 'Tongji1_CY25-05_1--5', 'Tongji1_CY25-05_1--6', 'Tongji1_CY25-05_1--7', 'Tongji1_CY25-05_1--8', 'Tongji1_CY25-05_1--9', 'Tongji1_CY25-1_1--1', 'Tongji1_CY25-1_1--2', 'Tongji1_CY25-1_1--3', 'Tongji1_CY25-1_1--4', 'Tongji1_CY25-1_1--5', 'Tongji1_CY25-1_1--6', 'Tongji1_CY25-1_1--7', 'Tongji1_CY25-1_1--8', 'Tongji1_CY25-1_1--9', 'Tongji1_CY35-05_1--1', 'Tongji1_CY35-05_1--2', 'Tongji1_CY35-05_1--3', 'Tongji1_CY45-05_1--1', 'Tongji1_CY45-05_1--10', 'Tongji1_CY45-05_1--11', 'Tong

In [32]:
from scipy.signal import find_peaks
import numpy as np

v_shape_features = []

for data in datasets:

    for cycle in data["cycle_data"]:

        v = np.array(cycle["voltage_in_V"], dtype=np.float32)

        dv = np.diff(v)
        d2v = np.diff(dv)

        peaks, _ = find_peaks(v)

        v_shape_features.append({
            "cell_id": data["cell_id"],
            "cycle_number": cycle["cycle_number"],

            "V_range": np.max(v) - np.min(v),
            "V_slope_mean": np.mean(np.abs(dv)),
            "V_curvature": np.mean(np.abs(d2v)),
            "V_n_peaks": len(peaks)
        })

v_shape_df = pd.DataFrame(v_shape_features)

In [33]:
feature_df = feature_df.merge(
    v_shape_df,
    on=["cell_id", "cycle_number"],
    how="left"
)

In [34]:
print(feature_df.columns.tolist())
print(feature_df[
    [
        "V_range",
        "V_slope_mean",
        "V_curvature",
        "V_n_peaks"
    ]
].head())

['cell_id', 'cycle_number', 'SOH', 'I_mean', 'I_std', 'charge_duration', 'discharge_duration', 'V_mean', 'V_std', 'DTW_I', 'DTW_V', 'V_range', 'V_slope_mean', 'V_curvature', 'V_n_peaks']
    V_range  V_slope_mean  V_curvature  V_n_peaks
0  1.700266      0.003730     0.001472         99
1  1.700266      0.003731     0.001473        107
2  1.700187      0.003736     0.001463         99
3  1.700423      0.003727     0.001458         99
4  1.700305      0.003743     0.001453        101


In [35]:
with open("processed_battery_features_v2.pkl", "wb") as f:
    pickle.dump(feature_df, f)

In [36]:
# with open("processed_battery_features_v2.pkl", "rb") as f:
#     feature_dftest = pickle.load(f)

In [37]:
# print(feature_dftest.shape)
# feature_dftest.head()